In [0]:
from pyspark.sql import functions as F

df = spark.read.table("edtech_project.edtech_silver.student_course_engagement")

In [0]:
df = df.withColumn(
    "risk_tier",
    F.when(
        (F.col("days_since_last_active") > 14) &
        (F.col("video_completion_rate") < 30),
        "HIGH"
    ).when(
        (F.col("days_since_last_active") > 7) |
        (F.col("quiz_pass_rate") < 0.4) |
        (F.col("video_completion_rate") < 0.25),
        "MEDIUM"
    ).otherwise("LOW")
)

In [0]:

df = df.withColumn("week", F.weekofyear(F.col("last_active_date")))


In [0]:
from pyspark.sql.window import Window

w = Window.partitionBy("student_id", "course_id").orderBy("week")

df = df.withColumn(
    "prev_risk",
    F.lag("risk_tier").over(w)
    
)

df = df.withColumn(
    "risk_worsened",
    F.when(
        ((F.col("prev_risk") == "LOW") & (F.col("risk_tier") == "MEDIUM")) |
        ((F.col("prev_risk") == "MEDIUM") & (F.col("risk_tier") == "HIGH")),
        1
    ).otherwise(0)
)

In [0]:
# There is no error in this code snippet. It correctly writes the DataFrame to a Delta table.
df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("edtech_project.edtech_silver.enrollment_risk_profile")

In [0]:
display(df)